# Chime — Train Your Own Personal Brain (Colab Free T4)

This notebook fine-tunes the **80M-parameter 24k-vocab Chime Causal Reranker** on your sanitized personal corpus (`chime sanitize input.txt clean_corpus.txt`) using **LoRA (`r=16, alpha=32`)** mixed with 30% general bilingual replay data to prevent catastrophic forgetting, then exports both:
1. `model.safetensors` for Apple Silicon `mlx-rs` unified-memory batch scoring
2. `ChimeReranker.mlpackage` with macOS 15 `MLState` stateful KV-cache for ANE execution
3. `personal.tsv` mined domain terms & code-switching phrases

In [ ]:
!pip install -q torch transformers peft datasets safetensors pypinyin coremltools

In [ ]:
from collections import Counter
from pathlib import Path
import re
from pypinyin import lazy_pinyin

CORPUS_PATH = Path('clean_corpus.txt')
lines = CORPUS_PATH.read_text(encoding='utf-8').splitlines() if CORPUS_PATH.exists() else [
    '今天在 macOS 上用 Rust 和 Swift 写双拼输入法',
    '帮我 review 一下这个 PR 的 tail latency',
]

# 1. Mine frequent personal English/code identifiers & Chinese phrases
eng_tokens = Counter(re.findall(r'[A-Za-z][A-Za-z0-9_.-]{2,}', '\n'.join(lines)))
with open('personal.tsv', 'w', encoding='utf-8') as f:
    for tok, freq in eng_tokens.most_common(500):
        f.write(f'{tok}\t{tok.lower()}\t{freq * 10}\n')
print(f'Exported {min(len(eng_tokens), 500)} personal code-switching terms to personal.tsv')

In [ ]:
# 2. LoRA configuration (r=16, alpha=32) for 80M Qwen2.5-distilled reranker
from peft import LoraConfig, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
print('Ready for LoRA training on T4:', lora_cfg)